<h2 align='center'>ML Flow and Dagshub</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

#### Handle class imbalance

In [6]:
!pip install --upgrade scikit-learn

In [7]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

### Track Experiments

In [8]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [9]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [10]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [11]:
!pip install dagshub

In [12]:
# dagshub setup

import dagshub
dagshub.init(repo_owner='lemanhtrung', repo_name='mlflow_dagshub', mlflow=True)

Accessing as lemanhtrung

Initialized MLflow to track repo "lemanhtrung/mlflow_dagshub"

Repository lemanhtrung/mlflow_dagshub initialized!

In [22]:
# Ideally you will not require following 4 lines if you have started fresh and do not have any previous dagshub credentials on your computer
# import os
# os.environ['MLFLOW_TRACKING_USERNAME'] = 'lemanhtrung' 
# os.environ['MLFLOW_TRACKING_PASSWORD'] = '595e6484b7ecfb7c988380ef7360fd54c1c21ccf' # 
# os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow' 

# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
# mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_tracking_uri("https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, 'xgb_model')
        else:
            mlflow.sklearn.log_model(sk_model=model, artifact_path=model_name)

2025/07/10 13:10:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0/runs/bf5a1b211eec4a198180caac52df0f14
🧪 View experiment at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0


2025/07/10 13:10:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0/runs/e277ad4e990e4487896b8ea5f2868b6c
🧪 View experiment at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0


2025/07/10 13:10:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0/runs/8e76b1b9242a464297e1c9ac7c51b215
🧪 View experiment at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0


2025/07/10 13:11:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0/runs/159a73be96a74c179052872d97d48b50
🧪 View experiment at: https://dagshub.com/lemanhtrung/mlflow_dagshub.mlflow/#/experiments/0
